# PolluVital AI Training Notebook (Colab)
This notebook trains:
1. rPPG backbone (PhysNet-style)
2. Breathing sound classifier (Librosa + CNN/LSTM)
3. Multimodal fusion LSTM (1–4h degradation risk)
4. Exports quantized TensorFlow Lite models

In [ ]:
!python --version
!pip -q install tensorflow==2.15.0 librosa scikit-learn pandas numpy matplotlib

In [ ]:
import os
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow.keras import layers, models
from sklearn.model_selection import train_test_split

## 1) Load Labels

In [ ]:
labels = pd.read_csv('labels_sample.csv')
labels.head()

## 2) rPPG Model (PhysNet-style placeholder)
For production, adapt architectures from PhysNet/EfficientPhys and train on UBFC/PURE + your Delhi set.

In [ ]:
def build_rppg_model():
    inp = layers.Input(shape=(150, 36, 36, 3), name='video_clip')
    x = layers.TimeDistributed(layers.Conv2D(16, 3, activation='relu', padding='same'))(inp)
    x = layers.TimeDistributed(layers.MaxPool2D())(x)
    x = layers.TimeDistributed(layers.Conv2D(32, 3, activation='relu', padding='same'))(x)
    x = layers.TimeDistributed(layers.GlobalAveragePooling2D())(x)
    x = layers.Bidirectional(layers.LSTM(32, return_sequences=False))(x)
    hr = layers.Dense(1, name='hr')(x)
    rr = layers.Dense(1, name='rr')(x)
    spo2 = layers.Dense(1, activation='sigmoid', name='spo2')(x)
    stress = layers.Dense(1, activation='sigmoid', name='stress')(x)
    model = models.Model(inp, [hr, rr, spo2, stress], name='rppg_model')
    model.compile(optimizer='adam', loss='mse')
    return model

rppg_model = build_rppg_model()
rppg_model.summary()

## 3) Audio Breathing Model (Librosa features + CNN)

In [ ]:
def build_audio_model(n_classes=3):
    inp = layers.Input(shape=(128, 64, 1), name='mel')
    x = layers.Conv2D(16, 3, activation='relu', padding='same')(inp)
    x = layers.MaxPool2D()(x)
    x = layers.Conv2D(32, 3, activation='relu', padding='same')(x)
    x = layers.MaxPool2D()(x)
    x = layers.Flatten()(x)
    x = layers.Dense(64, activation='relu')(x)
    out = layers.Dense(n_classes, activation='softmax', name='breath_class')(x)
    model = models.Model(inp, out, name='audio_model')
    model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
    return model

audio_model = build_audio_model()
audio_model.summary()

## 4) Fusion LSTM (Vitals + audio probs + AQI -> 1h/2h/4h risk)

In [ ]:
def build_fusion_model(seq_len=12, feat_dim=10):
    inp = layers.Input(shape=(seq_len, feat_dim), name='fusion_seq')
    x = layers.LSTM(64, return_sequences=True)(inp)
    x = layers.LSTM(32)(x)
    risk_1h = layers.Dense(1, activation='sigmoid', name='risk_1h')(x)
    risk_2h = layers.Dense(1, activation='sigmoid', name='risk_2h')(x)
    risk_4h = layers.Dense(1, activation='sigmoid', name='risk_4h')(x)
    hr_delta = layers.Dense(3, name='hr_delta_1_2_4h')(x)
    model = models.Model(inp, [risk_1h, risk_2h, risk_4h, hr_delta], name='fusion_lstm')
    model.compile(optimizer='adam', loss='mse')
    return model

fusion_model = build_fusion_model()
fusion_model.summary()

## 5) Quantization + TFLite Export

In [ ]:
def export_tflite(model, save_path):
    converter = tf.lite.TFLiteConverter.from_keras_model(model)
    converter.optimizations = [tf.lite.Optimize.DEFAULT]
    tflite_model = converter.convert()
    with open(save_path, 'wb') as f:
        f.write(tflite_model)
    print('Saved:', save_path)

os.makedirs('export', exist_ok=True)
export_tflite(rppg_model, 'export/rppg_physnet.tflite')
export_tflite(audio_model, 'export/audio_breath_cnn.tflite')
export_tflite(fusion_model, 'export/fusion_lstm.tflite')

## 6) Evaluation Target
- HR error target: ±5 bpm
- Fusion risk target: >=82% on Delhi split

Track MAE, F1, ROC-AUC with a fixed held-out Delhi/NCR set.